# NLP - Processamento de Linguagem Natural

**Tarefa:** _[Classificação / Sentimento / NER / Similaridade / Embeddings]_  
**Dataset:** _[nome / fonte]_  
**Idioma:** Português

## 1. Imports

In [ ]:
import re
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

nltk.download(['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'rslp'], quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

STOPWORDS = set(stopwords.words('portuguese'))
stemmer   = RSLPStemmer()

sns.set_theme(style='whitegrid')

## 2. Carregamento

In [ ]:
# df = pd.read_csv('textos.csv')

# Exemplo sintético para testar o pipeline:
textos_exemplo = [
    'O produto é excelente, recomendo muito!',
    'Péssima experiência, nunca mais compro.',
    'Entrega rápida e produto de boa qualidade.',
    'Não gostei do atendimento ao cliente.',
]
df = pd.DataFrame({'texto': textos_exemplo, 'label': [1, 0, 1, 0]})
df.head()

## 3. Pré-processamento

In [ ]:
def limpar_texto(texto: str) -> str:
    texto = texto.lower()
    texto = re.sub(r'http\S+', '', texto)           # remove URLs
    texto = re.sub(r'[^a-záéíóúàâêôãõç ]', '', texto)  # só letras PT
    tokens = texto.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

def aplicar_stemming(texto: str) -> str:
    return ' '.join([stemmer.stem(t) for t in texto.split()])


df['texto_limpo']  = df['texto'].apply(limpar_texto)
df['texto_stem']   = df['texto_limpo'].apply(aplicar_stemming)
df[['texto', 'texto_limpo', 'texto_stem']].head()

## 4. Análise do Vocabulário

In [ ]:
todos_tokens = ' '.join(df['texto_limpo']).split()
freq = Counter(todos_tokens)
top20 = pd.DataFrame(freq.most_common(20), columns=['token', 'freq'])

top20.plot(kind='barh', x='token', y='freq', figsize=(8, 5), legend=False)
plt.title('Top 20 Tokens mais Frequentes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Vetorização (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df['texto_limpo'])
print(f'Matriz TF-IDF: {X_tfidf.shape}')

## 6. Embeddings Semânticos (Sentence Transformers)

In [ ]:
from sentence_transformers import SentenceTransformer

# Modelo multilíngue leve, funciona bem em português
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = modelo_emb.encode(df['texto'].tolist(), show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')

## 7. Similaridade Semântica

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(6, 5))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f'T{i}' for i in range(len(df))],
            yticklabels=[f'T{i}' for i in range(len(df))])
plt.title('Similaridade Coseno entre Textos')
plt.tight_layout()
plt.show()

## 8. Classificação de Texto

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Usando embeddings como features
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df['label'], test_size=0.3, random_state=42
)

clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))

## 9. Conclusões

- _Qualidade dos embeddings para o domínio_
- _Performance do classificador_
- _Próximos passos (fine-tuning, modelos maiores, aumento de dados)_